# Cost waterfall over an OpenTelemetry trace

This notebook answers one question for a single agent conversation:

> this conversation cost $X. if you had routed the cheap turns to a smaller model
> it would have cost $Y. when is the switch worth it?

The input is a plain OTLP/JSON trace export (the format any OpenTelemetry exporter
emits), with spans following the GenAI semantic conventions. No provider SDK, no
vendor client, no lock-in: swap the trace file and the price table and it runs on
anyone's stack.

Steps: load the trace, build a per-call cost table, draw the waterfall, attribute
the retry spend, run a counterfactual reroute to Haiku, and compute the switch-when
breakeven.

In [1]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.width", 120)

from llm_trace_cost import (
    load_otlp, llm_calls, cost_table, waterfall,
    retry_breakdown, counterfactual, breakeven_retry_rate, DEFAULT_PRICES,
)

spans = load_otlp("../data/sample_trace.json")
print(f"loaded {len(spans)} spans from the OTLP export")

loaded 7 spans from the OTLP export


## 1. Per-call cost table

Each GenAI span priced from its own token counts against the price table. The
cumulative column is the x-axis of the waterfall.

In [2]:
df = llm_calls(spans, DEFAULT_PRICES)
ct = cost_table(df)
print(ct[["turn", "operation", "model", "input_tokens", "output_tokens",
       "is_retry", "cost_usd", "cum_cost_usd"]].round(4).to_string(index=False))

 turn operation         model  input_tokens  output_tokens  is_retry  cost_usd  cum_cost_usd
    0      chat claude-opus-4          8200            620     False    0.1695        0.1695
    1      tool claude-opus-4           340             70     False    0.0104        0.1798
    2      chat claude-opus-4          2100            410     False    0.0622        0.2421
    3      chat claude-opus-4          2250             40      True    0.0368        0.2788
    3      chat claude-opus-4          2250            380     False    0.0622        0.3411
    4      tool claude-opus-4           280             65     False    0.0091        0.3502
    5      chat claude-opus-4          1500            300     False    0.0450        0.3952


## 2. The waterfall

Cost grouped by turn, with each turn's share of the bill. The opening turn carries
the system prompt and full context, so it dominates. An ASCII bar keeps the output
copy-pasteable into a terminal or a PR comment.

In [3]:
wf = waterfall(df).round(4)
maxc = wf["cost_usd"].max()
for _, r in wf.iterrows():
    bar = "#" * int(round(r["cost_usd"] / maxc * 40))
    print(f"turn {int(r['turn']):>2}  ${r['cost_usd']:>7.4f}  {r['pct_of_total']:>5.1f}%  {bar}")
print(f"{'total':>7}  ${wf['cost_usd'].sum():>7.4f}")

turn  0  $ 0.1695   42.9%  ########################################
turn  1  $ 0.0104    2.6%  ##
turn  2  $ 0.0622   15.8%  ###############
turn  3  $ 0.0990   25.1%  #######################
turn  4  $ 0.0091    2.3%  ##
turn  5  $ 0.0450   11.4%  ###########
  total  $ 0.3952


## 3. Retry attribution

The genuinely under-instrumented dimension. A failed attempt that gets retried is
spend with no output. Reading `gen_ai.request.retry_count` off the spans isolates
how much of the bill is wasted attempts. (That attribute is not yet in the published
OTel conventions; the scope issue proposes standardizing it. When it is absent the
tool treats the call as a first attempt, so nothing breaks on a stock exporter.)

In [4]:
rb = retry_breakdown(df)
print(f"total spend   : ${rb['total_usd']:.4f}")
print(f"retry spend   : ${rb['retry_usd']:.4f}  ({rb['retry_pct']:.1f}% of the bill)")
print(f"retried calls : {rb['n_retries']}")

total spend   : $0.3952
retry spend   : $0.0367  (9.3% of the bill)
retried calls : 1


## 4. Counterfactual: route the cheap turns to Haiku

A routing policy reassigns a model per call. Here: send tool calls and any short
turn (under 150 output tokens) to Haiku, keep the heavier reasoning turns on Opus.
Cost is recomputed from the same token counts, the standard first-order assumption
for this question.

In [5]:
def policy(row):
    if row["operation"] == "tool" or row["output_tokens"] < 150:
        return ("anthropic", "claude-haiku-3-5")
    return None

cf = counterfactual(df, DEFAULT_PRICES, policy)
print(cf[["turn", "operation", "model", "cf_model", "cost_usd", "cf_cost_usd", "saving_usd"]].round(4).to_string(index=False))

 turn operation         model         cf_model  cost_usd  cf_cost_usd  saving_usd
    0      chat claude-opus-4    claude-opus-4    0.1695       0.1695      0.0000
    1      tool claude-opus-4 claude-haiku-3-5    0.0104       0.0006      0.0098
    2      chat claude-opus-4    claude-opus-4    0.0622       0.0622      0.0000
    3      chat claude-opus-4 claude-haiku-3-5    0.0368       0.0020      0.0348
    3      chat claude-opus-4    claude-opus-4    0.0622       0.0622      0.0000
    4      tool claude-opus-4 claude-haiku-3-5    0.0091       0.0005      0.0086
    5      chat claude-opus-4    claude-opus-4    0.0450       0.0450      0.0000


In [6]:
base_total = df["cost_usd"].sum()
cf_total = cf["cf_cost_usd"].sum()
saving = base_total - cf_total
print(f"baseline (all Opus) : ${base_total:.4f}")
print(f"routed              : ${cf_total:.4f}")
print(f"saving / conversation: ${saving:.4f}  ({saving / base_total * 100:.1f}%)")
print()
for v in (10_000, 50_000, 250_000):
    print(f"@ {v:>7,} conversations/mo: baseline ${base_total*v:>10,.0f}   routed ${cf_total*v:>10,.0f}   saved ${saving*v:>9,.0f}")

baseline (all Opus) : $0.3952
routed              : $0.3420
saving / conversation: $0.0532  (13.5%)

@  10,000 conversations/mo: baseline $     3,952   routed $     3,420   saved $      532
@  50,000 conversations/mo: baseline $    19,759   routed $    17,100   saved $    2,659
@ 250,000 conversations/mo: baseline $    98,794   routed $    85,499   saved $   13,295


## 5. Switch-when

Routing saves money only if the cheap model does not fail so often that the redos
eat the saving. Per rerouted turn with Opus cost `o` and Haiku cost `h`, expected
saving is `(o - h) - p * o` for failure rate `p`, so the breakeven is
`p* = saving / redo_cost = 1 - h/o`. Route while the expected failure rate is below
`p*`; above it, keep Opus.

In [7]:
redo_cost = cf.loc[cf["saving_usd"] > 0, "cost_usd"].sum()
p_star = breakeven_retry_rate(saving, redo_cost)
print(f"saving / conversation : ${saving:.4f}")
print(f"redo cost (Opus)      : ${redo_cost:.4f}")
print(f"breakeven failure rate: {p_star * 100:.1f}%")
print()
print(f"verdict: route the cheap turns unless you expect the cheap model to fail")
print(f"and need an Opus redo more than {p_star * 100:.0f}% of the time on those turns.")

saving / conversation : $0.0532
redo cost (Opus)      : $0.0562
breakeven failure rate: 94.7%

verdict: route the cheap turns unless you expect the cheap model to fail
and need an Opus redo more than 95% of the time on those turns.


## Takeaways

- the opening context-loading turn is where the money goes, not the chatty turns.
- ~9% of this bill is a single retried attempt: retry spend is real and usually invisible.
- routing tool calls and short turns to Haiku cuts ~13% with a breakeven failure rate
  near 95%, i.e. an easy call here.
- everything ran off a standard OTLP export and an editable price table. point it at
  your own trace and your own contract prices and the same four numbers come out.